# Diarios de avances de exploración
¿Cuánto aporta cada algoritmo y en qué momento conviene compartir cálculos? Cada sección ejecuta una variante en el mismo puzzle pequeño y después compara las mediciones reales con el paso anterior. Menos instrucciones es mejor.

Las comparaciones usan únicamente IDs resueltos por ambos métodos. La tabla final incluye cobertura para distinguir resultados parciales de entregas completas. Los 1.000 puntuables se miden localmente.

## Datos y funciones de lectura
Se toma la corrida completa más reciente de cada conjunto. Para regenerar los reportes desde la raíz, ejecutar `python3 -m scripts.compare_dag` (práctica) o `python3 -m scripts.scored_dag` (puntuables). Ejecutar el diario solo lee esos reportes y resuelve el ejemplo.

In [1]:
import csv
from pathlib import Path
from scripts.compare_dag import (Puzzle, validate, incremental, rings, bisection, multirings,
    bisection_reuse, multirings_reuse, postprocess, optimize_a4, nested_windows, optimize_dag)

def leer(ruta):
    with ruta.open(newline='') as archivo:
        return list(csv.DictReader(archivo))

def reporte(prefijo, marca):
    archivos = list(Path('reports').glob(f'{prefijo}_*/{marca}'))
    if not archivos:
        raise FileNotFoundError(f'Falta una corrida completa de {prefijo}. Ver comandos arriba.')
    return max(archivos, key=lambda p: p.stat().st_mtime_ns).parent

carpetas = {'Práctica': reporte('dag_practice', 'summary.csv'),
            'Puntuables': reporte('dag_scored', 'submission.csv')}
datos = {k: leer(v / 'comparison.csv') for k, v in carpetas.items()}
ejemplo = Puzzle('ejemplo', [1, 3, 10], [27, 97, 99, 107], 200)
historial = []

def medir(filas, columna, algoritmo=None, etapa=None):
    valores = {}
    for fila in filas:
        if fila['error'] or (algoritmo and fila['algorithm'] != algoritmo):
            continue
        if etapa and fila['insertion'] != etapa:
            continue
        valor = int(fila[columna])
        k = fila['id']
        valores[k] = min(valores.get(k, valor), valor)
    return valores

def comparar(nombre, columna, algoritmo=None, etapa=None):
    actual = {k: medir(filas, columna, algoritmo, etapa) for k, filas in datos.items()}
    for k, valores in actual.items():
        if historial:
            prev = historial[-1][1][k]
            comunes = prev.keys() & valores.keys()
            ahorro = sum(prev[i] - valores[i] for i in comunes)
            print(f'{k}: {len(valores)} resueltos | {len(comunes)} comunes | ahorro frente a {historial[-1][0]}: {ahorro:+,}')
        else:
            print(f'{k}: {len(valores)} resueltos | {sum(valores.values()):,} instrucciones')
    historial.append((nombre, actual))

for k, ruta in carpetas.items():
    print(k, ruta)

Práctica reports/dag_practice_gpvvc01y
Puntuables reports/dag_scored_ojhhopuu


## A1 · Incremental
Conserva un uno y suma hasta alcanzar cada objetivo en orden. Sirve como primera referencia y puede agotar el límite de instrucciones.

In [2]:
programa = incremental(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 107, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Establece la referencia inicial.

In [3]:
comparar('A1 · Incremental', 'before', 'incremental', 'constructor')

Práctica: 16 resueltos | 144 instrucciones
Puntuables: 16 resueltos | 139 instrucciones


## A2 · Anillos
Avanza con múltiplos de una base y completa el resto para alcanzar el objetivo.

In [4]:
programa = rings(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 42, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [5]:
comparar('A2 · Anillos', 'before', 'rings', 'constructor')

Práctica: 159 resueltos | 16 comunes | ahorro frente a A1 · Incremental: +8
Puntuables: 778 resueltos | 16 comunes | ahorro frente a A1 · Incremental: +4


## A3 · Bisección
Parte del intervalo entre múltiplos y construye valores intermedios para acercarse al objetivo.

In [6]:
programa = bisection(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 42, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [7]:
comparar('A3 · Bisección', 'before', 'bisection', 'constructor')

Práctica: 142 resueltos | 133 comunes | ahorro frente a A2 · Anillos: -126,549
Puntuables: 775 resueltos | 684 comunes | ahorro frente a A2 · Anillos: -683,353


## Múltiples anillos
Explora distintas bases iniciales para acortar la construcción.

In [8]:
programa = multirings(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 33, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [9]:
comparar('Múltiples anillos', 'before', 'multirings', 'constructor')

Práctica: 163 resueltos | 142 comunes | ahorro frente a A3 · Bisección: +67,763
Puntuables: 853 resueltos | 775 comunes | ahorro frente a A3 · Bisección: +369,669


## Bisección con reutilización
Aprovecha extremos y valores ya presentes en registros durante la bisección.

In [10]:
programa = bisection_reuse(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 20, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [11]:
comparar('Bisección con reutilización', 'before', 'bisection_reuse', 'constructor')

Práctica: 200 resueltos | 163 comunes | ahorro frente a Múltiples anillos: +153,313
Puntuables: 1000 resueltos | 853 comunes | ahorro frente a Múltiples anillos: +829,456


## Múltiples anillos con reutilización
Combina la elección de base con el aprovechamiento de valores residentes. Este programa alimenta las tres pasadas siguientes.

In [12]:
programa = multirings_reuse(ejemplo)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 14, 'covered': 4, 'valid': True}


### Medición en ambos conjuntos
Compara el constructor con el de la sección anterior sobre los IDs comunes.

In [13]:
comparar('Múltiples anillos con reutilización', 'before', 'multirings_reuse', 'constructor')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Bisección con reutilización: +14,062
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Bisección con reutilización: +66,480


## Posprocesamiento
Comprime operaciones y elimina cálculos sin uso.

In [14]:
programa = postprocess(ejemplo, programa)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 9, 'covered': 4, 'valid': True}


### Efecto de la pasada
Mide el ahorro respecto del programa de la etapa anterior.

In [15]:
comparar('Posprocesamiento', 'before', 'multirings_reuse', 'postprocess')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Múltiples anillos con reutilización: +114,905
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Múltiples anillos con reutilización: +576,348


## A4
Busca atajos que permiten pasarse de un objetivo y corregir por resta o división exacta.

In [16]:
programa = optimize_a4(ejemplo, programa)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 9, 'covered': 4, 'valid': True}


### Efecto de la pasada
Mide el ahorro respecto del programa de la etapa anterior.

In [17]:
comparar('A4', 'before', 'multirings_reuse', 'a4')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Posprocesamiento: +550
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Posprocesamiento: +2,579


## Ventanas de ventanas
Une bloques vecinos para aprovechar valores que cruzan sus límites.

In [18]:
programa = nested_windows(ejemplo, programa)
print(validate(ejemplo, programa))

{'id': 'ejemplo', 'steps': 9, 'covered': 4, 'valid': True}


### Efecto de la pasada
Mide el ahorro respecto del programa de la etapa anterior.

In [19]:
comparar('Ventanas de ventanas', 'before', 'multirings_reuse', 'nested')

Práctica: 200 resueltos | 200 comunes | ahorro frente a A4: +437
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a A4: +2,239


## Mejor control por puzzle
Cada constructor válido recibe las tres pasadas. Se elige el programa final más corto por ID para medir el efecto de poder cambiar de constructor.

In [20]:
constructores = (incremental, rings, bisection, multirings, bisection_reuse, multirings_reuse)
pasadas = (postprocess, optimize_a4, nested_windows)

def ruta(constructor, insercion=None):
    programa = constructor(ejemplo)
    for posicion in range(4):
        if posicion == insercion:
            programa = optimize_dag(ejemplo, programa)
        if posicion < len(pasadas):
            programa = pasadas[posicion](ejemplo, programa)
    return validate(ejemplo, programa)['steps']

print(min(ruta(fn) for fn in constructores))

7


### Comparación con ventanas
Cuantifica el cambio de usar el mejor constructor por ID.

In [21]:
comparar('Mejor control sin DAG', 'control_final')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Ventanas de ventanas: +9,800
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Ventanas de ventanas: +44,318


## DAG después de constructor
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [22]:
print(min(ruta(fn, 0) for fn in constructores))

10


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [23]:
comparar('DAG después de constructor', 'dag_final', etapa='constructor')

Práctica: 200 resueltos | 200 comunes | ahorro frente a Mejor control sin DAG: +41,649
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a Mejor control sin DAG: +194,289


## DAG después de posprocesamiento
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [24]:
print(min(ruta(fn, 1) for fn in constructores))

7


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [25]:
comparar('DAG después de posprocesamiento', 'dag_final', etapa='postprocess')

Práctica: 200 resueltos | 200 comunes | ahorro frente a DAG después de constructor: -26,811
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a DAG después de constructor: -122,126


## DAG después de A4
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [26]:
print(min(ruta(fn, 2) for fn in constructores))

7


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [27]:
comparar('DAG después de A4', 'dag_final', etapa='a4')

Práctica: 200 resueltos | 200 comunes | ahorro frente a DAG después de posprocesamiento: -77
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a DAG después de posprocesamiento: -499


## DAG después de ventanas
Comparte valores iguales en el programa de esta etapa y ejecuta las pasadas pendientes. Se selecciona por ID el mejor de los constructores en ese orden.

In [28]:
print(min(ruta(fn, 3) for fn in constructores))

7


### Comparación del resultado final
Compara los programas terminados con la sección anterior.

In [29]:
comparar('DAG después de ventanas', 'dag_final', etapa='nested')

Práctica: 200 resueltos | 200 comunes | ahorro frente a DAG después de A4: +0
Puntuables: 1000 resueltos | 1000 comunes | ahorro frente a DAG después de A4: +0


## Selección final por puzzle
Se conserva el mejor de los controles y las cuatro posiciones del DAG. Para el entregable también compite el programa del CSV previo en cada ID.

In [30]:
seleccion = {}
for k, filas in datos.items():
    control = medir(filas, 'control_final')
    dag = medir(filas, 'dag_final')
    seleccion[k] = {i: min(valor, dag[i]) for i, valor in control.items()}
ganadores = leer(carpetas['Puntuables'] / 'winners.csv')
entrega = {r['id']: int(r['steps']) for r in ganadores}
anterior = {r['id']: int(r['previous']) for r in ganadores}
assert len(entrega) == 1000 and len(seleccion['Práctica']) == 200
assert entrega == {i: min(anterior[i], seleccion['Puntuables'][i]) for i in entrega}
print('Entregable validado:', carpetas['Puntuables'] / 'submission.csv')

Entregable validado: reports/dag_scored_ojhhopuu/submission.csv


### Comparación con la entrega anterior
Cuenta cuántos puzzles mejoraron y cuánto bajó la suma de instrucciones.

In [31]:
print('Puzzles mejorados:', sum(entrega[i] < anterior[i] for i in entrega))
print('Instrucciones ahorradas:', sum(anterior.values()) - sum(entrega.values()))

Puzzles mejorados: 978
Instrucciones ahorradas: 250357


## Tabla de avances
Totales de instrucciones calculados desde los reportes seleccionados. Cada fila indica cuántos puzzles se resolvieron sobre 200 o 1.000; una suma parcial solo describe esos IDs. El signo — indica que falta una medición para ese conjunto.

In [32]:
def celda(valores, cantidad):
    if not valores:
        return '—'
    total = f'{sum(valores.values()):,}'.replace(',', '.')
    return f'{total} ({len(valores)}/{cantidad})'

filas = historial + [
    ('Mejor control o DAG por ID', seleccion),
    ('Entregable anterior', {'Puntuables': anterior}),
    ('Entregable validado', {'Puntuables': entrega})]
from pathlib import Path
import csv

def cargar_pasos(ruta):
    with Path(ruta).open() as archivo:
        return {r["id"]: int(r["final"]) for r in csv.DictReader(archivo)}

rutas_practica = cargar_pasos("reports/partial_practice_i0l79tbp/winners.csv")
rutas_puntuables = cargar_pasos("reports/partial_scored_2jzgqmyl/winners.csv")
con_constructor = cargar_pasos("reports/backward_practice_wyyedfh3/comparison.csv")
assert rutas_practica.keys() == con_constructor.keys()
assert all(con_constructor[k] <= pasos for k, pasos in rutas_practica.items())
constructor_puntuables = cargar_pasos("reports/backward_scored_LECwzt0R/comparison.csv")
assert constructor_puntuables.keys() == rutas_puntuables.keys()
assert all(constructor_puntuables[k] <= pasos for k, pasos in rutas_puntuables.items())
nuevas_filas = [
    ("Cartera de cuatro rutas e histórico", {"Práctica": rutas_practica, "Puntuables": rutas_puntuables}),
    ("Cartera con constructor hacia atrás", {"Práctica": con_constructor, "Puntuables": constructor_puntuables}),
]
filas.extend(nuevas_filas)
from html import escape
from IPython.display import HTML, display

encabezados = ('Algoritmo o entrega', 'Práctica · pasos (cobertura)', 'Puntuables · pasos (cobertura)')
tabla = '<table><thead><tr>' + ''.join(f'<th>{escape(t)}</th>' for t in encabezados) + '</tr></thead><tbody>'
for nombre, valores in filas:
    columnas = (nombre, celda(valores.get('Práctica'), 200), celda(valores.get('Puntuables'), 1000))
    tabla += '<tr>' + ''.join(f'<td>{escape(v)}</td>' for v in columnas) + '</tr>'
display(HTML(tabla + '</tbody></table>'))

Algoritmo o entrega,Práctica · pasos (cobertura),Puntuables · pasos (cobertura)
A1 · Incremental,144 (16/200),139 (16/1000)
A2 · Anillos,234.714 (159/200),1.166.626 (778/1000)
A3 · Bisección,332.967 (142/200),1.929.943 (775/1000)
Múltiples anillos,340.301 (163/200),1.834.813 (853/1000)
Bisección con reutilización,281.552 (200/200),1.372.573 (1000/1000)
Múltiples anillos con reutilización,267.490 (200/200),1.306.093 (1000/1000)
Posprocesamiento,152.585 (200/200),729.745 (1000/1000)
A4,152.035 (200/200),727.166 (1000/1000)
Ventanas de ventanas,151.598 (200/200),724.927 (1000/1000)
Mejor control sin DAG,141.798 (200/200),680.609 (1000/1000)


## Cuatro rutas de búsqueda parcial en práctica
A conserva el original. B elige por longitud inicial y C posprocesa las mismas opciones antes de elegir. D busca sobre los originales ya posprocesados. Cada selección desempata en favor del original. La columna de pasos toma el mínimo por ID entre los seis constructores. Los tiempos suman búsqueda y optimización por constructor, atribuyendo a cada ruta el trabajo reutilizado. Excluyen construcción, indexación de donantes y reconstrucción histórica.
La [tabla por ID](reports/partial_practice_i0l79tbp/winners.csv) registra el ganador frente al histórico. El [detalle por constructor y ruta](reports/partial_practice_i0l79tbp/comparison.csv) incluye fallos de cobertura, índice del candidato (0 para el original) y longitudes por etapa. En D la entrada corresponde al candidato formado después del posprocesamiento.

In [33]:
from pathlib import Path
import csv
from scripts.validate_submission import verify

carpeta = Path("reports/partial_practice_i0l79tbp")
def leer(nombre):
    with (carpeta / nombre).open() as archivo:
        return list(csv.DictReader(archivo))

detalle = [r for r in leer("comparison.csv") if not r["error"]]
ganadores = leer("winners.csv")
validos, total = verify(Path("practice_puzzles.csv"), carpeta / "programs.csv")
por_ruta = {}
print("Ruta   IDs   Candidatos   Pasos   Segundos acumulados")
for ruta in "ABCD":
    filas = [r for r in detalle if r["route"] == ruta]
    mejores = {}
    for r in filas:
        k, pasos = r["id"], int(r["final"])
        mejores[k] = min(mejores.get(k, pasos), pasos)
    por_ruta[ruta] = mejores
    print(ruta, len(mejores), sum(int(r["candidates"]) for r in filas),
          sum(mejores.values()), round(sum(float(r["seconds"]) for r in filas), 1))

for nueva, control in [("C", "B"), ("D", "A")]:
    delta = [por_ruta[control][k] - v for k, v in por_ruta[nueva].items()]
    print(f"{nueva} frente a {control}: {sum(d > 0 for d in delta)} mejoras,"
          f" {delta.count(0)} empates, {sum(d < 0 for d in delta)} retrocesos,"
          f" {sum(delta)} pasos ahorrados.")
    pares = {(r["id"], r["algorithm"]): int(r["final"]) for r in detalle if r["route"] == control}
    cambios = [pares[r["id"], r["algorithm"]] - int(r["final"]) for r in detalle if r["route"] == nueva]
    print("Por par ID-constructor (mejor / igual / peor):", sum(d > 0 for d in cambios),
          cambios.count(0), sum(d < 0 for d in cambios))

previo = sum(int(r["historical"]) for r in ganadores)
assert validos == len(ganadores) == 200 and previo == 98127
for r in ganadores:
    assert int(r["final"]) == min(int(r["historical"]), *(s[r["id"]] for s in por_ruta.values()))
assert total == sum(int(r["final"]) for r in ganadores)
print(f"Cartera validada: {total} pasos en {validos} puzzles. Ahorro histórico: {previo-total}.")
print("IDs mejorados:", sum(int(r["final"]) < int(r["historical"]) for r in ganadores))


Ruta   IDs   Candidatos   Pasos   Segundos acumulados
A 200 880 141798 2948.0
B 200 6988 103295 2786.1
C 200 6988 100957 22459.0
D 200 6953 128287 2974.4
C frente a B: 147 mejoras, 52 empates, 1 retrocesos, 2338 pasos ahorrados.
Por par ID-constructor (mejor / igual / peor): 602 270 8
D frente a A: 173 mejoras, 26 empates, 1 retrocesos, 13511 pasos ahorrados.
Por par ID-constructor (mejor / igual / peor): 733 146 1
Cartera validada: 92881 pasos en 200 puzzles. Ahorro histórico: 5246.
IDs mejorados: 145


## Búsqueda hacia atrás y recombinación de recetas

En la versión revisada de royalacecat, combinar los mejores programas públicos por ID ya suma 132.623 pasos. La búsqueda seleccionada para `scored_0295` lleva ese puzzle de 119 a 118 y el total a 132.622. Nuestro programa tiene 366 pasos en ese ID. La mayor diferencia ya existe al comenzar esa búsqueda. [Notebook de royalacecat, versión 372165224](https://www.kaggle.com/code/royalacecat/santa-2026-public-frontier-score-132-638?scriptVersionId=372165224)

GoodPJW v42 registra 132.636 pasos para la unión pública y 132.634 al incorporar dos programas propios. Su búsqueda local elimina una rama de dependencias y la reconstruye con valores del resto del grafo. Guarda recetas de programas empatados. CP-SAT combina esas recetas, elige un orden válido y permite que dos objetivos compartan un intermedio. El autor reporta ahorros de 45, 28 y 4 pasos en tres pasadas anteriores. Los puntajes públicos son resultados de sus autores. [Notebook de GoodPJW, v42](https://www.kaggle.com/code/goodpjw2008/santa-2026-lns-cp-sat-recombine-lb-132-634?scriptVersionId=372161473)

### Qué hace el constructor de Tony Li

Para cada objetivo prepara un diccionario de expresiones que cuestan hasta dos operaciones desde valores de partida, constantes, objetivos ya alcanzados e intermedios recientes. Si el objetivo no aparece, busca descomposiciones por cociente y resto. Por ejemplo, con 16 y 1 disponibles, construir 97 se reduce a construir 6 mediante la identidad `97 = 6 × 16 + 1`. El haz conserva varias reducciones prometedoras y repite la búsqueda sobre sus cocientes. Después emite solo las dependencias necesarias, asigna registros y puede reconstruir un valor desalojado. Konbu17 amplía el diccionario y combina varias configuraciones por ID. Su total publicado de 142.940 también incluye otros métodos. [Tony Li](https://www.kaggle.com/code/tonylica/elf-calculator-with-register-reuse?scriptVersionId=370866029) · [Konbu17](https://www.kaggle.com/code/konbu17/santa-2026-wider-dictionary-portfolio-163k?scriptVersionId=371106642)

### Experimento sobre los 200 puzzles de práctica

Se compilaron las tres celdas C++ originales de Tony Li con `g++ -std=c++17 -O3`. Cada corrida recibe los mismos objetivos ordenados y solo cambia el ancho del haz (1, 3 o 6). El valor publicado es 3. Cada programa parte de los valores iniciales del puzzle. La referencia local es la mejor cartera por ID después de las cuatro rutas (92.881 instrucciones). La celda siguiente reproduce la comparación desde los programas guardados. La elección del menor resultado por ID requiere ejecutar todas las configuraciones que se comparan. [Fuente compilada](reports/backward_practice_wyyedfh3/solver.cpp)


In [34]:
import csv
import json
from pathlib import Path
from scripts.validate_submission import verify

datos = Path("practice_puzzles.csv")
carpeta = Path("reports/backward_practice_wyyedfh3")
archivos = {
    "Cartera anterior": Path("reports/partial_practice_i0l79tbp/programs.csv"),
    "Ancho 1": carpeta / "programs_w1.csv",
    "Ancho 3": carpeta / "programs.csv",
    "Ancho 6": carpeta / "programs_w6.csv",
}
programas = {}
for nombre, ruta in archivos.items():
    cantidad, total = verify(datos, ruta)
    assert cantidad == 200
    with ruta.open() as fp:
        programas[nombre] = {r["id"]: r["program"] for r in csv.DictReader(fp)}
    print(f"{nombre:16} {total:>7} pasos  ({cantidad}/200)")

with datos.open() as fp:
    puzzles_practica = list(csv.DictReader(fp))
familias = {}
mejores_tres = 0
ganancias = empates = 0
with (carpeta / "comparison.csv").open("w", newline="") as tabla, (carpeta / "portfolio.csv").open("w", newline="") as salida:
    detalle, entrega = csv.writer(tabla), csv.writer(salida)
    detalle.writerow(["id", "familia", *archivos, "ganador", "final"])
    entrega.writerow(["id", "program"])
    for puzzle in puzzles_practica:
        k, familia = puzzle["id"], puzzle["family"]
        pasos = {n: len(json.loads(banco[k])) for n, banco in programas.items()}
        previo = pasos["Cartera anterior"]
        nuevo = min(pasos[n] for n in ("Ancho 1", "Ancho 3", "Ancho 6"))
        mejores_tres += nuevo
        ganancias += nuevo < previo
        empates += nuevo == previo
        ganador = min(pasos, key=pasos.get)
        detalle.writerow([k, familia, *pasos.values(), ganador, pasos[ganador]])
        entrega.writerow([k, programas[ganador][k]])
        suma = familias.setdefault(familia, [0, 0, 0])
        suma[0] += 1
        suma[1] += previo
        suma[2] += nuevo

print(f"Mejor de los tres anchos por ID: {mejores_tres} pasos.")
print(f"Frente al anterior: {ganancias} mejoras, {empates} empates y {200-ganancias-empates} retrocesos.")
for familia, (cuantos, anterior, atras) in sorted(familias.items()):
    print(f"{familia:15} {cuantos:3} puzzles   {anterior:6} → {atras:6}")
cantidad, total = verify(datos, carpeta / "portfolio.csv")
print(f"Cartera combinada validada: {cantidad} programas, {total} pasos.")

Cartera anterior   92881 pasos  (200/200)
Ancho 1            40474 pasos  (200/200)
Ancho 3            39135 pasos  (200/200)
Ancho 6            38648 pasos  (200/200)
Mejor de los tres anchos por ID: 38566 pasos.
Frente al anterior: 183 mejoras, 3 empates y 14 retrocesos.
independent      45 puzzles    32715 →  12973
mixed            48 puzzles    27815 →  11471
multiplicative   40 puzzles    10396 →   5244
shared           51 puzzles    21901 →   8801
tiny             16 puzzles       54 →     77
Cartera combinada validada: 200 programas, 38540 pasos.


### Aporte del constructor a la cartera de práctica
Al sumar los anchos 1, 3 y 6 como opciones por ID, la cartera pasa de 92.881 a 38.540 instrucciones (54.341 menos, 58,51 %). Mejoran 183 IDs y 17 mantienen su longitud. En 14 de esos 17 el programa anterior es estrictamente más corto; en los otros 3 empata. La mayor reducción absoluta aparece en la familia independiente (19.742 instrucciones). En los 16 puzzles diminutos, la selección conserva el total previo de 54. La medición del nuevo constructor cubre los 200 puzzles de práctica.

### Constructor hacia atrás sobre los 1.000 puntuables

| Configuración | Instrucciones |
|---|---:|
| Cartera anterior | 448.063 |
| Ancho 1 | 197.890 |
| Ancho 3 | 191.867 |
| Ancho 6 | 189.141 |
| Cartera combinada | 188.917 |

El mejor de los tres anchos mejora 965 IDs, empata en 11 y queda por detrás en 24. La selección conserva la cartera anterior en los empates. El verificador reprodujo los 1.000 programas de cada configuración y los del CSV final. [Comparación por ID](reports/backward_scored_LECwzt0R/comparison.csv) · [Entrega validada](artifacts/submission_backward_188917.csv)